In [33]:
# ============================================================================
# IMPORTS
# ============================================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.metrics import classification_report, roc_curve, auc
from sklearn.base import clone
from xgboost import XGBClassifier, XGBRegressor

In [34]:
# ============================================================================
# 1. LOAD DATA
# ============================================================================

# Load scaled and unscaled feature sets (sorted by video_id)
x_train_scaled = pd.read_csv("classifier/integrated_features/data/x_train_scaled.csv").sort_values("video_id")
x_test_scaled = pd.read_csv("classifier/integrated_features/data/x_test_scaled.csv").sort_values("video_id")

x_train_unscaled = pd.read_csv("classifier/integrated_features/data/x_train_unscaled.csv").sort_values("video_id")
x_test_unscaled = pd.read_csv("classifier/integrated_features/data/x_test_unscaled.csv").sort_values("video_id")

# Load labels (continuous scores)
y_train = pd.read_csv("classifier/integrated_features/data/train_y.csv").sort_values("video_id")
y_test = pd.read_csv("classifier/integrated_features/data/test_y.csv").sort_values("video_id")

print("Shapes:")
print(f"x_train_scaled: {x_train_scaled.shape}")
print(f"x_test_scaled:  {x_test_scaled.shape}")
print(f"y_train:        {y_train.shape}")
print(f"y_test:         {y_test.shape}")

Shapes:
x_train_scaled: (133, 32)
x_test_scaled:  (49, 32)
y_train:        (140, 2)
y_test:         (49, 2)


In [35]:
# ============================================================================
# 2. PREPARE LABELS: CONTINUOUS SCORE + BINARY LABELS
# ============================================================================

# Identify score column (the second column if video_id exists)
label_col = y_train.columns[1] if "video_id" in y_train.columns else y_train.columns[0]

# Extract continuous scores
y_train_score = y_train[label_col].values
y_test_score = y_test[label_col].values

# Binary labels using threshold = 0.8
threshold = 0.8
y_train_binary = (y_train_score >= threshold).astype(int)
y_test_binary = (y_test_score >= threshold).astype(int)

print("\nLabel column:", label_col)
print(f"Train score range: {y_train_score.min():.3f} - {y_train_score.max():.3f}")
print(f"Test score range:  {y_test_score.min():.3f} - {y_test_score.max():.3f}")

print("\nBinary label distribution:")
print("Train:", pd.Series(y_train_binary).value_counts().sort_index())
print("Test:", pd.Series(y_test_binary).value_counts().sort_index())



Label column: label
Train score range: 0.444 - 1.000
Test score range:  0.422 - 1.000

Binary label distribution:
Train: 0    57
1    83
Name: count, dtype: int64
Test: 0    20
1    29
Name: count, dtype: int64


In [36]:
# ============================================================================
# 3. ALIGN DATA BY video_id
# ============================================================================

print("\nAligning data by video_id...")

# Determine common video_ids between feature and label sets
train_ids_x = set(x_train_scaled["video_id"].unique())
test_ids_x = set(x_test_scaled["video_id"].unique())
train_ids_y = set(y_train["video_id"].unique())
test_ids_y = set(y_test["video_id"].unique())

train_common = sorted(list(train_ids_x & train_ids_y))
test_common = sorted(list(test_ids_x & test_ids_y))

print(f"Common train video_ids: {len(train_common)}")
print(f"Common test video_ids:  {len(test_common)}")

# Filter and align by video_id
x_train_aligned = (
    x_train_scaled[x_train_scaled["video_id"].isin(train_common)]
    .sort_values("video_id")
    .reset_index(drop=True)
)

x_test_aligned = (
    x_test_scaled[x_test_scaled["video_id"].isin(test_common)]
    .sort_values("video_id")
    .reset_index(drop=True)
)

y_train_aligned = (
    y_train[y_train["video_id"].isin(train_common)]
    .sort_values("video_id")
    .reset_index(drop=True)
)

y_test_aligned = (
    y_test[y_test["video_id"].isin(test_common)]
    .sort_values("video_id")
    .reset_index(drop=True)
)

# Build feature matrices
X_train = x_train_aligned.drop(columns=["video_id"]).values
X_test = x_test_aligned.drop(columns=["video_id"]).values
feature_names = x_train_aligned.drop(columns=["video_id"]).columns.tolist()

# Aligned labels
y_train_score_aligned = y_train_aligned[label_col].values
y_test_score_aligned = y_test_aligned[label_col].values

# Align binary labels
y_train_binary_aligned = (
    pd.DataFrame({"video_id": y_train["video_id"], "label": y_train_binary})
    .loc[lambda df: df["video_id"].isin(train_common)]
    .sort_values("video_id")["label"]
    .values
)

y_test_binary_aligned = (
    pd.DataFrame({"video_id": y_test["video_id"], "label": y_test_binary})
    .loc[lambda df: df["video_id"].isin(test_common)]
    .sort_values("video_id")["label"]
    .values
)

print("\nVerification:")
print("X_train:", X_train.shape)
print("y_train_score_aligned:", y_train_score_aligned.shape)
print("y_train_binary_aligned:", y_train_binary_aligned.shape)

print("X_test:", X_test.shape)
print("y_test_score_aligned:", y_test_score_aligned.shape)
print("y_test_binary_aligned:", y_test_binary_aligned.shape)


Aligning data by video_id...
Common train video_ids: 133
Common test video_ids:  48

Verification:
X_train: (133, 31)
y_train_score_aligned: (133,)
y_train_binary_aligned: (133,)
X_test: (48, 31)
y_test_score_aligned: (48,)
y_test_binary_aligned: (48,)


In [10]:
# ============================================================================
# 4. FUNCTION: REGRESSION + THRESHOLD SEARCH
# ============================================================================

def run_regression_with_threshold(
    model,
    model_name,
    X_train,
    y_train_score,
    y_train_bin,
    X_test,
    y_test_score,
    y_test_bin,
    thr_min=0.3,
    thr_max=0.9,
    thr_step=0.01,
    n_splits=5,
    random_state=42,
):
    """
    Full pipeline:
    1. Train regression model using cross-val and generate OOF predictions.
    2. Search best threshold on OOF predictions for binary classification.
    3. Train final model on full training data.
    4. Evaluate on both train and test sets using tuned threshold.
    """
    print("\n" + "=" * 80)
    print(f"MODEL: {model_name}")
    print("=" * 80)

    # ---- STEP 1: Generate Out-of-Fold (OOF) predictions ----
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    oof_scores = np.zeros_like(y_train_score, dtype=float)

    for fold_idx, (tr_idx, val_idx) in enumerate(cv.split(X_train, y_train_bin), 1):
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr = y_train_score[tr_idx]

        m = clone(model)
        m.fit(X_tr, y_tr)
        oof_scores[val_idx] = m.predict(X_val)

        print(f"  Fold {fold_idx} completed.")

    # ---- STEP 2: Search best threshold ----
    thresholds = np.arange(thr_min, thr_max + thr_step, thr_step)
    best_thr = None
    best_acc = -1
    best_f1 = -1

    for thr in thresholds:
        preds = (oof_scores >= thr).astype(int)
        acc = accuracy_score(y_train_bin, preds)
        f1 = f1_score(y_train_bin, preds)

        # Use accuracy as primary metric, F1 as tie-breaker
        if acc > best_acc or (np.isclose(acc, best_acc) and f1 > best_f1):
            best_thr = thr
            best_acc = acc
            best_f1 = f1

    print("\nBest threshold found:")
    print(f"  threshold = {best_thr:.3f}")
    print(f"  CV accuracy = {best_acc:.4f}")
    print(f"  CV F1       = {best_f1:.4f}")

    # ---- STEP 3: Fit final regression model on full training set ----
    final_model = clone(model)
    final_model.fit(X_train, y_train_score)

    # Predictions
    train_score_pred = final_model.predict(X_train)
    test_score_pred = final_model.predict(X_test)

    # Convert to binary labels using tuned threshold
    train_bin_pred = (train_score_pred >= best_thr).astype(int)
    test_bin_pred = (test_score_pred >= best_thr).astype(int)

    # ---- STEP 4: Compute metrics ----
    def report(split_name, y_true_bin, score_pred, bin_pred):
        acc = accuracy_score(y_true_bin, bin_pred)
        prec = precision_score(y_true_bin, bin_pred, zero_division=0)
        rec = recall_score(y_true_bin, bin_pred, zero_division=0)
        f1 = f1_score(y_true_bin, bin_pred, zero_division=0)
        auc = roc_auc_score(y_true_bin, score_pred)

        print(f"\n{split_name} Performance (threshold = {best_thr:.3f}):")
        print(f"  Accuracy:  {acc:.4f}")
        print(f"  Precision: {prec:.4f}")
        print(f"  Recall:    {rec:.4f}")
        print(f"  F1:        {f1:.4f}")
        print(f"  AUC:       {auc:.4f}")

        return dict(accuracy=acc, precision=prec, recall=rec, f1=f1, auc=auc)

    train_metrics = report("Train", y_train_bin, train_score_pred, train_bin_pred)
    test_metrics = report("Test", y_test_bin, test_score_pred, test_bin_pred)

    return dict(
        model=final_model,
        best_threshold=best_thr,
        train_metrics=train_metrics,
        test_metrics=test_metrics,
        train_score_pred=train_score_pred,
        test_score_pred=test_score_pred,
    )


In [37]:
# ============================================================================
# 5. RUN RIDGE REGRESSION
# ============================================================================

ridge_model = Ridge(alpha=1.0, random_state=42)

ridge_results = run_regression_with_threshold(
    model=ridge_model,
    model_name="Ridge Regression",
    X_train=X_train,
    y_train_score=y_train_score_aligned,
    y_train_bin=y_train_binary_aligned,
    X_test=X_test,
    y_test_score=y_test_score_aligned,
    y_test_bin=y_test_binary_aligned,
)

print("\nRidge Regression completed.")
print("Best threshold:", ridge_results["best_threshold"])
print("Test metrics:", ridge_results["test_metrics"])



MODEL: Ridge Regression
  Fold 1 completed.
  Fold 2 completed.
  Fold 3 completed.
  Fold 4 completed.
  Fold 5 completed.

Best threshold found:
  threshold = 0.750
  CV accuracy = 0.6241
  CV F1       = 0.7396

Train Performance (threshold = 0.750):
  Accuracy:  0.6316
  Precision: 0.6311
  Recall:    0.9506
  F1:        0.7586
  AUC:       0.7424

Test Performance (threshold = 0.750):
  Accuracy:  0.5208
  Precision: 0.5750
  Recall:    0.7931
  F1:        0.6667
  AUC:       0.4828

Ridge Regression completed.
Best threshold: 0.7500000000000004
Test metrics: {'accuracy': 0.5208333333333334, 'precision': 0.575, 'recall': 0.7931034482758621, 'f1': 0.6666666666666666, 'auc': np.float64(0.4827586206896552)}


In [ ]:
# ============================================================================
# RIDGE REGRESSION: Visualization and Detailed Analysis
# ============================================================================

print("=" * 80)
print("RIDGE REGRESSION: Visualization and Detailed Analysis")
print("=" * 80)

# ---------------------------------------------------------------------------
# 1. Confusion Matrix and Classification Report (Test Set)
# ---------------------------------------------------------------------------

# Assumes ridge_results contains binary predictions for the test set
y_test_pred_ridge = ridge_results["y_test_pred"]
cm_test_ridge = confusion_matrix(y_test_binary_aligned, y_test_pred_ridge)

print("\nClassification Report (Test Set):")
print("=" * 80)
print(classification_report(y_test_binary_aligned, y_test_pred_ridge, target_names=["Low", "High"]))

# ---------------------------------------------------------------------------
# 2. Visualizations
# ---------------------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Confusion Matrix
ax1 = axes[0, 0]
sns.heatmap(
    cm_test_ridge,
    annot=True,
    fmt="d",
    cmap="Blues",
    ax=ax1,
    cbar_kws={"shrink": 0.8},
    xticklabels=["Low", "High"],
    yticklabels=["Low", "High"],
)
ax1.set_xlabel("Predicted", fontsize=12)
ax1.set_ylabel("Actual", fontsize=12)
ax1.set_title("Ridge Regression: Confusion Matrix (Test Set)", fontsize=12, fontweight="bold")

# Plot 2: ROC Curve
ax2 = axes[0, 1]
y_test_proba_ridge = ridge_results.get("y_test_proba", None)

if y_test_proba_ridge is not None:
    fpr_ridge, tpr_ridge, _ = roc_curve(y_test_binary_aligned, y_test_proba_ridge)
    roc_auc_ridge = auc(fpr_ridge, tpr_ridge)

    ax2.plot(fpr_ridge, tpr_ridge, lw=2, label=f"ROC curve (AUC = {roc_auc_ridge:.3f})")
    ax2.plot([0, 1], [0, 1], lw=2, linestyle="--", label="Random")
    ax2.set_xlabel("False Positive Rate", fontsize=12)
    ax2.set_ylabel("True Positive Rate", fontsize=12)
    ax2.set_title("Ridge Regression: ROC Curve (Test Set)", fontsize=12, fontweight="bold")
    ax2.legend(loc="lower right")
    ax2.grid(alpha=0.3)
else:
    ax2.text(
        0.5,
        0.5,
        "ROC curve not available",
        ha="center",
        va="center",
        fontsize=12,
        transform=ax2.transAxes,
    )
    ax2.set_title("Ridge Regression: ROC Curve (Not Available)", fontsize=12, fontweight="bold")

# Plot 3: Coefficient Importance (Top 15 by absolute value)
ax3 = axes[1, 0]

# Assumes ridge_results contains the fitted model or you can reuse ridge_model if it was fit in-place
best_ridge_model = ridge_results.get("best_model", ridge_model)

coeffs = np.ravel(best_ridge_model.coef_)
abs_coeffs = np.abs(coeffs)

# If you already have feature_names defined (as in the decision tree block)
feature_names_ridge = feature_names

# Sort coefficients by absolute value
sorted_idx = np.argsort(abs_coeffs)[::-1]
top_n = min(15, len(feature_names_ridge))
top_idx = sorted_idx[:top_n]

ax3.barh(range(top_n), abs_coeffs[top_idx], alpha=0.7)
ax3.set_yticks(range(top_n))
ax3.set_yticklabels([feature_names_ridge[i][:30] for i in top_idx], fontsize=9)
ax3.invert_yaxis()
ax3.set_xlabel("Absolute Coefficient", fontsize=12)
ax3.set_title(f"Ridge Regression: Top {top_n} Coefficients", fontsize=12, fontweight="bold")
ax3.grid(axis="x", alpha=0.3)

# Plot 4: Train vs Test Performance Comparison
ax4 = axes[1, 1]

# Assumes ridge_results["train_metrics"] and ridge_results["test_metrics"] are dicts with these keys
train_metrics_ridge = ridge_results.get("train_metrics", {})
test_metrics_ridge = ridge_results.get("test_metrics", {})

metrics = ["accuracy", "precision", "recall", "f1"]
metric_labels = ["Accuracy", "Precision", "Recall", "F1"]

train_values = [train_metrics_ridge.get(m, np.nan) for m in metrics]
test_values = [test_metrics_ridge.get(m, np.nan) for m in metrics]

x_pos = np.arange(len(metrics))
width = 0.35

ax4.bar(x_pos - width / 2, train_values, width, label="Train", alpha=0.7)
ax4.bar(x_pos + width / 2, test_values, width, label="Test", alpha=0.7)

ax4.set_xticks(x_pos)
ax4.set_xticklabels(metric_labels, fontsize=10)
ax4.set_ylabel("Score", fontsize=12)
ax4.set_title("Ridge Regression: Train vs Test Performance", fontsize=12, fontweight="bold")
ax4.legend()
ax4.grid(axis="y", alpha=0.3)
ax4.set_ylim([0, 1.1])

# Add value labels on bars
for i, (train_val, test_val) in enumerate(zip(train_values, test_values)):
    if not np.isnan(train_val):
        ax4.text(i - width / 2, min(train_val + 0.02, 1.05), f"{train_val:.3f}", ha="center", va="bottom", fontsize=8)
    if not np.isnan(test_val):
        ax4.text(i + width / 2, min(test_val + 0.02, 1.05), f"{test_val:.3f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

# ---------------------------------------------------------------------------
# 3. Detailed Coefficient Summary
# ---------------------------------------------------------------------------

print(f"\nAll Features and Coefficients ({len(feature_names_ridge)}):")
print("=" * 80)
for i, (name, coef) in enumerate(zip(feature_names_ridge, coeffs), 1):
    print(f"{i:2d}. {name:40s} : {coef:.6f}")

print("\nTop 10 Features by Absolute Coefficient:")
print("=" * 80)
top_10_idx = sorted_idx[:10]
for i, idx in enumerate(top_10_idx, 1):
    print(f"{i:2d}. {feature_names_ridge[idx]:40s} : coef = {coeffs[idx]: .6f} (|coef| = {abs_coeffs[idx]:.6f})")

print("\n" + "=" * 80)


In [12]:
# ============================================================================
# 6. RANDOM FOREST REGRESSION
# ============================================================================
#
rf_reg = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
)

rf_results = run_regression_with_threshold(
    model=rf_reg,
    model_name="RandomForest Regressor",
    X_train=X_train,
    y_train_score=y_train_score_aligned,
    y_train_bin=y_train_binary_aligned,
    X_test=X_test,
    y_test_score=y_test_score_aligned,
    y_test_bin=y_test_binary_aligned,
)

print("\nRandomForest completed.")
print("Best threshold:", rf_results["best_threshold"])
print("Test metrics:", rf_results["test_metrics"])


MODEL: RandomForest Regressor
  Fold 1 completed.
  Fold 2 completed.
  Fold 3 completed.
  Fold 4 completed.
  Fold 5 completed.

Best threshold found:
  threshold = 0.300
  CV accuracy = 0.6090
  CV F1       = 0.7570

Train Performance (threshold = 0.300):
  Accuracy:  0.6090
  Precision: 0.6090
  Recall:    1.0000
  F1:        0.7570
  AUC:       0.9934

Test Performance (threshold = 0.300):
  Accuracy:  0.6042
  Precision: 0.6042
  Recall:    1.0000
  F1:        0.7532
  AUC:       0.5608

RandomForest completed.
Best threshold: 0.3
Test metrics: {'accuracy': 0.6041666666666666, 'precision': 0.6041666666666666, 'recall': 1.0, 'f1': 0.7532467532467533, 'auc': np.float64(0.5607985480943739)}


In [14]:
# ============================================================================
# 2. DEFINE SCORE → 3-CLASS MAPPING
# ============================================================================

# Identify the score column name
label_col = y_train.columns[1] if "video_id" in y_train.columns else y_train.columns[0]

# Define thresholds for 3-class labeling
low_thr = 0.65
high_thr = 0.80

def score_to_3class(score, low=low_thr, high=high_thr):
    """
    Map continuous score to 3 ordinal classes:
      0: Low    (score < low)
      1: Medium (low <= score < high)
      2: High   (score >= high)
    """
    if score < low:
        return 0
    elif score < high:
        return 1
    else:
        return 2

# Quick view of raw score ranges
print("\nRaw score ranges:")
print(f"Train: {y_train[label_col].min():.3f} - {y_train[label_col].max():.3f}")
print(f"Test:  {y_test[label_col].min():.3f} - {y_test[label_col].max():.3f}")

# ============================================================================
# 3. ALIGN BY video_id
# ============================================================================

print("\nAligning data by video_id...")

train_ids_x = set(x_train_scaled["video_id"].unique())
test_ids_x = set(x_test_scaled["video_id"].unique())
train_ids_y = set(y_train["video_id"].unique())
test_ids_y = set(y_test["video_id"].unique())

train_common = sorted(list(train_ids_x & train_ids_y))
test_common = sorted(list(test_ids_x & test_ids_y))

print(f"Common train video_ids: {len(train_common)}")
print(f"Common test  video_ids: {len(test_common)}")

# Align X
x_train_aligned = (
    x_train_scaled[x_train_scaled["video_id"].isin(train_common)]
    .sort_values("video_id")
    .reset_index(drop=True)
)
x_test_aligned = (
    x_test_scaled[x_test_scaled["video_id"].isin(test_common)]
    .sort_values("video_id")
    .reset_index(drop=True)
)

# Align y
y_train_aligned = (
    y_train[y_train["video_id"].isin(train_common)]
    .sort_values("video_id")
    .reset_index(drop=True)
)
y_test_aligned = (
    y_test[y_test["video_id"].isin(test_common)]
    .sort_values("video_id")
    .reset_index(drop=True)
)

# Feature matrices
X_train = x_train_aligned.drop(columns=["video_id"]).values
X_test = x_test_aligned.drop(columns=["video_id"]).values
feature_names = x_train_aligned.drop(columns=["video_id"]).columns.tolist()

# Continuous scores (aligned)
y_train_score = y_train_aligned[label_col].values
y_test_score = y_test_aligned[label_col].values

# 3-class labels (aligned)
y_train_3class = np.array([score_to_3class(s) for s in y_train_score], dtype=int)
y_test_3class = np.array([score_to_3class(s) for s in y_test_score], dtype=int)

# Binary labels for "High vs Others"
# High: class == 2, Others: class in {0, 1}
y_train_bin_high = (y_train_3class == 2).astype(int)
y_test_bin_high = (y_test_3class == 2).astype(int)

print("\nVerification:")
print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train_score:", y_train_score.shape)
print("y_test_score: ", y_test_score.shape)
print("y_train_3class:", np.bincount(y_train_3class))
print("y_test_3class: ", np.bincount(y_test_3class))
print("y_train_bin_high:", np.bincount(y_train_bin_high))
print("y_test_bin_high: ", np.bincount(y_test_bin_high))

# ============================================================================
# 4. HELPER: REPORT BINARY METRICS (High vs Others)
# ============================================================================

def report_binary_metrics(name, y_true_bin, score_pred, bin_pred):
    """
    Print and return binary classification metrics:
      accuracy, precision, recall, F1, AUC
    """
    acc = accuracy_score(y_true_bin, bin_pred)
    prec = precision_score(y_true_bin, bin_pred, zero_division=0)
    rec = recall_score(y_true_bin, bin_pred, zero_division=0)
    f1 = f1_score(y_true_bin, bin_pred, zero_division=0)
    auc = roc_auc_score(y_true_bin, score_pred)

    print(f"\n{name} performance (High vs Others):")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1:        {f1:.4f}")
    print(f"  AUC:       {auc:.4f}")

    return dict(accuracy=acc, precision=prec, recall=rec, f1=f1, auc=auc)

# ============================================================================
# 5. METHOD A: XGBoostClassifier (3-class softmax)
# ============================================================================

print("\n" + "=" * 80)
print("METHOD A: XGBoostClassifier (3-class) + High vs Others")
print("=" * 80)

# XGBoost multi-class classifier
clf = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42,
    tree_method="auto",  # adjust if GPU available
)

# Train on 3-class labels
clf.fit(X_train, y_train_3class)

# 3-class predictions
train_3class_pred = clf.predict(X_train)
test_3class_pred = clf.predict(X_test)

# 3-class accuracy (for reference)
train_3class_acc = accuracy_score(y_train_3class, train_3class_pred)
test_3class_acc = accuracy_score(y_test_3class, test_3class_pred)

print("\n3-class accuracy:")
print(f"  Train: {train_3class_acc:.4f}")
print(f"  Test:  {test_3class_acc:.4f}")

# Convert to binary High vs Others
train_bin_pred_A = (train_3class_pred == 2).astype(int)
test_bin_pred_A = (test_3class_pred == 2).astype(int)

# For AUC, we use the predicted probability of "High" (class index = 2)
train_proba = clf.predict_proba(X_train)[:, 2]
test_proba = clf.predict_proba(X_test)[:, 2]

metrics_A_train = report_binary_metrics(
    "METHOD A - Train", y_train_bin_high, train_proba, train_bin_pred_A
)
metrics_A_test = report_binary_metrics(
    "METHOD A - Test", y_test_bin_high, test_proba, test_bin_pred_A
)

# ============================================================================
# 6. METHOD B: XGBoostRegressor (ordinal target 0/1/2) + threshold search
# ============================================================================

print("\n" + "=" * 80)
print("METHOD B: XGBoostRegressor (ordinal 0/1/2) + threshold search")
print("=" * 80)

def run_ordinal_regression_with_threshold(
    model,
    X_train,
    y_train_ord,
    y_train_bin_high,
    X_test,
    y_test_ord,
    y_test_bin_high,
    thr_min=0.5,
    thr_max=2.5,
    thr_step=0.05,
    n_splits=5,
    random_state=42,
):
    """
    Use XGBoostRegressor with ordinal targets {0,1,2}.
    1. Generate out-of-fold predictions on train.
    2. Search best threshold for High vs Others based on binary labels.
    3. Fit final model and evaluate on train & test.
    """

    # Step 1: generate out-of-fold predictions
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    oof_pred = np.zeros_like(y_train_ord, dtype=float)

    for fold_idx, (tr_idx, val_idx) in enumerate(cv.split(X_train, y_train_bin_high), 1):
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr = y_train_ord[tr_idx]

        m = clone(model)
        m.fit(X_tr, y_tr)
        oof_pred[val_idx] = m.predict(X_val)

        print(f"  Fold {fold_idx} completed.")

    # Step 2: threshold search on OOF predictions (High vs Others)
    thresholds = np.arange(thr_min, thr_max + thr_step, thr_step)
    best_thr = None
    best_acc = -1.0
    best_f1 = -1.0

    for thr in thresholds:
        # High if prediction >= thr, otherwise Others
        bin_pred = (oof_pred >= thr).astype(int)
        acc = accuracy_score(y_train_bin_high, bin_pred)
        f1 = f1_score(y_train_bin_high, bin_pred)

        if acc > best_acc or (np.isclose(acc, best_acc) and f1 > best_f1):
            best_thr = thr
            best_acc = acc
            best_f1 = f1

    print("\nBest threshold from CV:")
    print(f"  threshold = {best_thr:.3f}")
    print(f"  CV accuracy = {best_acc:.4f}")
    print(f"  CV F1       = {best_f1:.4f}")

    # Step 3: fit final model on full training data
    final_model = clone(model)
    final_model.fit(X_train, y_train_ord)

    train_pred_cont = final_model.predict(X_train)
    test_pred_cont = final_model.predict(X_test)

    # Binary predictions with tuned threshold
    train_bin_pred = (train_pred_cont >= best_thr).astype(int)
    test_bin_pred = (test_pred_cont >= best_thr).astype(int)

    # Binary metrics (High vs Others)
    metrics_train = report_binary_metrics(
        "METHOD B - Train", y_train_bin_high, train_pred_cont, train_bin_pred
    )
    metrics_test = report_binary_metrics(
        "METHOD B - Test", y_test_bin_high, test_pred_cont, test_bin_pred
    )

    # Optional: 3-class metrics by rounding continuous predictions
    train_3class_pred = np.clip(np.rint(train_pred_cont), 0, 2).astype(int)
    test_3class_pred = np.clip(np.rint(test_pred_cont), 0, 2).astype(int)

    train_3class_acc = accuracy_score(y_train_ord, train_3class_pred)
    test_3class_acc = accuracy_score(y_test_ord, test_3class_pred)

    print("\n3-class accuracy from ordinal regression (rounded predictions):")
    print(f"  Train: {train_3class_acc:.4f}")
    print(f"  Test:  {test_3class_acc:.4f}")

    return dict(
        model=final_model,
        best_threshold=best_thr,
        train_metrics_bin=metrics_train,
        test_metrics_bin=metrics_test,
        train_3class_acc=train_3class_acc,
        test_3class_acc=test_3class_acc,
    )

# Define XGBRegressor for ordinal regression
reg = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method="auto",
)

results_B = run_ordinal_regression_with_threshold(
    model=reg,
    X_train=X_train,
    y_train_ord=y_train_3class.astype(float),  # ordinal targets {0,1,2}
    y_train_bin_high=y_train_bin_high,
    X_test=X_test,
    y_test_ord=y_test_3class.astype(float),
    y_test_bin_high=y_test_bin_high,
    thr_min=0.5,
    thr_max=2.5,
    thr_step=0.05,
    n_splits=5,
    random_state=42,
)

# ============================================================================
# 7. SUMMARY COMPARISON ON TEST SET
# ============================================================================

print("\n" + "=" * 80)
print("SUMMARY: High vs Others (Test Set)")
print("=" * 80)

print("\nMETHOD A: XGBoostClassifier")
for k, v in metrics_A_test.items():
    print(f"  {k}: {v:.4f}")

print("\nMETHOD B: XGBoostRegressor (ordinal + threshold)")
for k, v in results_B["test_metrics_bin"].items():
    print(f"  {k}: {v:.4f}")


Shapes:
x_train_scaled: (133, 32)
x_test_scaled:  (49, 32)
y_train:        (140, 2)
y_test:         (49, 2)

Raw score ranges:
Train: 0.444 - 1.000
Test:  0.422 - 1.000

Aligning data by video_id...
Common train video_ids: 133
Common test  video_ids: 48

Verification:
X_train: (133, 31)
X_test:  (48, 31)
y_train_score: (133,)
y_test_score:  (48,)
y_train_3class: [20 32 81]
y_test_3class:  [10  9 29]
y_train_bin_high: [52 81]
y_test_bin_high:  [19 29]

METHOD A: XGBoostClassifier (3-class) + High vs Others

3-class accuracy:
  Train: 1.0000
  Test:  0.5417

METHOD A - Train performance (High vs Others):
  Accuracy:  1.0000
  Precision: 1.0000
  Recall:    1.0000
  F1:        1.0000
  AUC:       1.0000

METHOD A - Test performance (High vs Others):
  Accuracy:  0.6042
  Precision: 0.6389
  Recall:    0.7931
  F1:        0.7077
  AUC:       0.5554

METHOD B: XGBoostRegressor (ordinal 0/1/2) + threshold search
  Fold 1 completed.
  Fold 2 completed.
  Fold 3 completed.
  Fold 4 completed.


In [15]:
# ============================================================================
# HELPER FUNCTIONS FOR 3-CLASS LABELS AND ORDINAL CV
# ============================================================================

import itertools

def make_3class_labels(scores, low_thr, high_thr):
    """
    Map continuous scores into 3 ordinal classes:
      0: Low    (score < low_thr)
      1: Medium (low_thr <= score < high_thr)
      2: High   (score >= high_thr)
    """
    labels = np.zeros_like(scores, dtype=int)
    labels[scores >= low_thr] = 1
    labels[scores >= high_thr] = 2
    return labels


def ordinal_cv_score_with_threshold(
    X,
    y_ord,
    y_bin_high,
    model,
    thr_min=0.5,
    thr_max=2.5,
    thr_step=0.05,
    n_splits=5,
    random_state=42,
):
    """
    Perform stratified K-fold CV for an ordinal regression model (targets 0/1/2):
      1. Fit model on K-1 folds, predict continuous scores on the held-out fold.
      2. Aggregate OOF predictions.
      3. Search best threshold for "High vs Others" classification.
      4. Return best threshold and corresponding CV metrics.

    Returns:
      best_thr, best_cv_acc, best_cv_f1
    """
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    oof_pred = np.zeros_like(y_ord, dtype=float)

    for fold_idx, (tr_idx, val_idx) in enumerate(cv.split(X, y_bin_high), 1):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr = y_ord[tr_idx]

        m = clone(model)
        m.fit(X_tr, y_tr)
        oof_pred[val_idx] = m.predict(X_val)

        # You can uncomment this if you want to see per-fold logs
        # print(f"  Fold {fold_idx} completed.")

    thresholds = np.arange(thr_min, thr_max + thr_step, thr_step)
    best_thr = None
    best_acc = -1.0
    best_f1 = -1.0

    for thr in thresholds:
        bin_pred = (oof_pred >= thr).astype(int)
        acc = accuracy_score(y_bin_high, bin_pred)
        f1 = f1_score(y_bin_high, bin_pred)

        if acc > best_acc or (np.isclose(acc, best_acc) and f1 > best_f1):
            best_thr = thr
            best_acc = acc
            best_f1 = f1

    return best_thr, best_acc, best_f1


In [16]:
# ============================================================================
# GRID SEARCH OVER (low_thr, high_thr) FOR 3-CLASS BUCKETS
# ============================================================================

low_grid = np.arange(0.55, 0.71, 0.02)   # e.g. 0.55, 0.57, ..., 0.69
high_grid = np.arange(0.75, 0.91, 0.02)  # e.g. 0.70, 0.72, ..., 0.90

min_samples_per_class = 5   # skip thresholds that yield too few samples in any class

best_pair = None
best_pair_thr = None
best_pair_cv_f1 = -1.0
best_pair_cv_acc = -1.0

results_grid = []

print("Starting grid search for (low_thr, high_thr)...\n")

for low_thr in low_grid:
    for high_thr in high_grid:
        # Ensure ordering and a reasonable gap between low and high
        if high_thr - low_thr < 0.05:
            continue

        # Build 3-class labels for train set
        y_train_3_tmp = make_3class_labels(y_train_score, low_thr, high_thr)
        counts = np.bincount(y_train_3_tmp)

        # Skip if any class is too small
        if len(counts) < 3 or counts.min() < min_samples_per_class:
            continue

        # Binary labels: High (class 2) vs Others
        y_train_bin_tmp = (y_train_3_tmp == 2).astype(int)
        bin_counts = np.bincount(y_train_bin_tmp)
        if bin_counts.min() < min_samples_per_class:
            continue

        # Define a base XGBoostRegressor (no heavy tuning yet)
        base_reg = XGBRegressor(
            objective="reg:squarederror",
            n_estimators=300,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            tree_method="auto",
        )

        best_thr, cv_acc, cv_f1 = ordinal_cv_score_with_threshold(
            X=X_train,
            y_ord=y_train_3_tmp.astype(float),
            y_bin_high=y_train_bin_tmp,
            model=base_reg,
            thr_min=0.5,
            thr_max=2.5,
            thr_step=0.05,
            n_splits=5,
            random_state=42,
        )

        results_grid.append(
            (low_thr, high_thr, best_thr, cv_acc, cv_f1, counts.tolist(), bin_counts.tolist())
        )

        # Use CV F1 as the primary objective, then CV accuracy
        if cv_f1 > best_pair_cv_f1 or (np.isclose(cv_f1, best_pair_cv_f1) and cv_acc > best_pair_cv_acc):
            best_pair = (low_thr, high_thr)
            best_pair_thr = best_thr
            best_pair_cv_f1 = cv_f1
            best_pair_cv_acc = cv_acc

            print(
                f"New best pair: low={low_thr:.3f}, high={high_thr:.3f}, "
                f"cv_acc={cv_acc:.4f}, cv_f1={cv_f1:.4f}, best_thr={best_thr:.3f}, "
                f"class_counts={counts}, bin_counts={bin_counts}"
            )

print("\nGrid search completed.")
print("Best (low_thr, high_thr):", best_pair)
print("Best CV F1:", best_pair_cv_f1)
print("Best CV accuracy:", best_pair_cv_acc)
print("Best internal threshold (High vs Others):", best_pair_thr)


Starting grid search for (low_thr, high_thr)...

New best pair: low=0.570, high=0.700, cv_acc=0.7669, cv_f1=0.8681, best_thr=0.500, class_counts=[  7  23 103], bin_counts=[ 30 103]
New best pair: low=0.610, high=0.700, cv_acc=0.7744, cv_f1=0.8729, best_thr=0.500, class_counts=[ 15  15 103], bin_counts=[ 30 103]

Grid search completed.
Best (low_thr, high_thr): (np.float64(0.6100000000000001), np.float64(0.7))
Best CV F1: 0.8728813559322034
Best CV accuracy: 0.7744360902255639
Best internal threshold (High vs Others): 0.5


In [27]:
# ============================================================================
# HYPERPARAMETER TUNING FOR XGBRegressor ON BEST (low_thr, high_thr)
# ============================================================================

# Use the best thresholds from previous grid search
best_low_thr, best_high_thr = best_pair

# Rebuild 3-class and binary labels with the best thresholds
y_train_3_best = make_3class_labels(y_train_score, best_low_thr, best_high_thr)
y_test_3_best = make_3class_labels(y_test_score, best_low_thr, best_high_thr)

y_train_bin_best = (y_train_3_best == 2).astype(int)
y_test_bin_best = (y_test_3_best == 2).astype(int)

print("\nUsing best thresholds:")
print(f"  low_thr  = {best_low_thr:.3f}")
print(f"  high_thr = {best_high_thr:.3f}")
print("Train 3-class counts:", np.bincount(y_train_3_best))
print("Test  3-class counts:", np.bincount(y_test_3_best))
print("Train binary (High vs Others):", np.bincount(y_train_bin_best))
print("Test  binary (High vs Others):", np.bincount(y_test_bin_best))

# Define a small hyperparameter grid for XGBRegressor
param_grid = {
    "max_depth": [2, 3, 4],
    "min_child_weight": [1, 3],
    "subsample": [0.7, 1.0],
    "colsample_bytree": [0.7, 1.0],
    "learning_rate": [0.03, 0.07],
}

best_params = None
best_model_thr = None
best_cv_f1_hp = -1.0
best_cv_acc_hp = -1.0

print("\nStarting hyperparameter search for XGBRegressor...\n")

for max_depth in param_grid["max_depth"]:
    for min_child_weight in param_grid["min_child_weight"]:
        for subsample in param_grid["subsample"]:
            for colsample_bytree in param_grid["colsample_bytree"]:
                for learning_rate in param_grid["learning_rate"]:
                    reg_hp = XGBRegressor(
                        objective="reg:squarederror",
                        n_estimators=400,
                        max_depth=max_depth,
                        min_child_weight=min_child_weight,
                        learning_rate=learning_rate,
                        subsample=subsample,
                        colsample_bytree=colsample_bytree,
                        random_state=42,
                        tree_method="auto",
                    )

                    best_thr_hp, cv_acc_hp, cv_f1_hp = ordinal_cv_score_with_threshold(
                        X=X_train,
                        y_ord=y_train_3_best.astype(float),
                        y_bin_high=y_train_bin_best,
                        model=reg_hp,
                        thr_min=0.5,
                        thr_max=2.5,
                        thr_step=0.05,
                        n_splits=5,
                        random_state=42,
                    )

                    # Update best hyperparams based on CV F1, then CV accuracy
                    if cv_f1_hp > best_cv_f1_hp or (
                        np.isclose(cv_f1_hp, best_cv_f1_hp) and cv_acc_hp > best_cv_acc_hp
                    ):
                        best_cv_f1_hp = cv_f1_hp
                        best_cv_acc_hp = cv_acc_hp
                        best_params = dict(
                            max_depth=max_depth,
                            min_child_weight=min_child_weight,
                            subsample=subsample,
                            colsample_bytree=colsample_bytree,
                            learning_rate=learning_rate,
                        )
                        best_model_thr = best_thr_hp

                        print(
                            f"New best params: {best_params}, "
                            f"CV acc={cv_acc_hp:.4f}, CV f1={cv_f1_hp:.4f}, "
                            f"best_thr={best_thr_hp:.3f}"
                        )

print("\nHyperparameter search completed.")
print("Best params:", best_params)
print("Best CV F1:", best_cv_f1_hp)
print("Best CV accuracy:", best_cv_acc_hp)
print("Best threshold (High vs Others):", best_model_thr)

# ============================================================================
# FINAL MODEL TRAINING AND EVALUATION WITH BEST SETTINGS
# ============================================================================

best_reg = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=400,
    max_depth=best_params["max_depth"],
    min_child_weight=best_params["min_child_weight"],
    learning_rate=best_params["learning_rate"],
    subsample=best_params["subsample"],
    colsample_bytree=best_params["colsample_bytree"],
    random_state=42,
    tree_method="auto",
)

# Fit on full training data with ordinal targets {0,1,2}
best_reg.fit(X_train, y_train_3_best.astype(float))

# Continuous predictions
train_pred_cont = best_reg.predict(X_train)
test_pred_cont = best_reg.predict(X_test)

# Binary predictions using tuned threshold
train_bin_pred_final = (train_pred_cont >= best_model_thr).astype(int)
test_bin_pred_final = (test_pred_cont >= best_model_thr).astype(int)

print("\nFinal performance with best thresholds and hyperparameters:")

final_train_metrics = report_binary_metrics(
    "FINAL - Train", y_train_bin_best, train_pred_cont, train_bin_pred_final
)
final_test_metrics = report_binary_metrics(
    "FINAL - Test", y_test_bin_best, test_pred_cont, test_bin_pred_final
)



Using best thresholds:
  low_thr  = 0.670
  high_thr = 0.770
Train 3-class counts: [28 15 90]
Test  3-class counts: [12  4 32]
Train binary (High vs Others): [43 90]
Test  binary (High vs Others): [16 32]

Starting hyperparameter search for XGBRegressor...

New best params: {'max_depth': 2, 'min_child_weight': 1, 'subsample': 0.7, 'colsample_bytree': 0.7, 'learning_rate': 0.03}, CV acc=0.6917, CV f1=0.8145, best_thr=0.650
New best params: {'max_depth': 2, 'min_child_weight': 1, 'subsample': 0.7, 'colsample_bytree': 1.0, 'learning_rate': 0.03}, CV acc=0.6992, CV f1=0.8182, best_thr=0.600
New best params: {'max_depth': 2, 'min_child_weight': 1, 'subsample': 1.0, 'colsample_bytree': 1.0, 'learning_rate': 0.03}, CV acc=0.7068, CV f1=0.8219, best_thr=0.500
New best params: {'max_depth': 3, 'min_child_weight': 3, 'subsample': 1.0, 'colsample_bytree': 1.0, 'learning_rate': 0.07}, CV acc=0.7143, CV f1=0.8257, best_thr=0.550

Hyperparameter search completed.
Best params: {'max_depth': 3, 'min_